# Hybrid Search

**Definition:** Hybrid search combines the semantic search from `003_vectordb` (finds meaning, misses exact tokens) with the lexical search from `004_bm25` (finds exact tokens, misses meaning) into one ranked result — giving you the strengths of both.

The technique used to merge two separately-ranked result lists into one is **Reciprocal Rank Fusion (RRF)**: for each document, take its *rank* (not its raw score) in each index's result list, and sum `1 / (k_rrf + rank)` across indexes. A document that ranks well in both lists — or even just one — bubbles to the top, without needing the two indexes' raw scores to be on comparable scales.

This notebook assumes `VectorIndex` (from `003_vectordb.ipynb`) and `BM25Index` (from `004_bm25.ipynb`) are already familiar — we paste them again below only so this notebook runs standalone, and focus the walkthrough on the one new piece: `Retriever`, which wraps both indexes and does the fusion.


In [1]:
%pip install -q voyageai python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Setup + chunking + embedding (from 002_embeddings)
from dotenv import load_dotenv
import voyageai

load_dotenv()

client = voyageai.Client()


def chunk_by_section(document_text):
    import re

    pattern = r"\n## "
    return re.split(pattern, document_text)


def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]


## The two indexes (pasted from 003 and 004)

Same `VectorIndex` and `BM25Index` classes as before — nothing new here. Skim past these; the interesting part is `Retriever`, right after.


In [3]:
# VectorIndex (from 003_vectordb) — semantic search
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(self, distance_metric: str = "cosine", embedding_fn=None):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_documents(self, documents, vectors=None):
        if vectors is None:
            vectors = self._embedding_fn([doc["content"] for doc in documents])
        for vector, document in zip(vectors, documents):
            self.add_vector(vector=vector, document=document)

    def add_document(self, document: Dict[str, Any]):
        vector = self._embedding_fn(document["content"])
        self.add_vector(vector=vector, document=document)

    def add_vector(self, vector, document: Dict[str, Any]):
        if not self.vectors:
            self._vector_dim = len(vector)
        self.vectors.append(list(vector))
        self.documents.append(document)

    def search(self, query: Any, k: int = 1):
        if not self.vectors:
            return []
        query_vector = self._embedding_fn(query) if isinstance(query, str) else query
        distances = [(self._cosine_distance(query_vector, v), doc) for v, doc in zip(self.vectors, self.documents)]
        distances.sort(key=lambda item: item[0])
        return [(doc, dist) for dist, doc in distances[:k]]

    def _cosine_distance(self, vec1, vec2) -> float:
        mag1 = math.sqrt(sum(x * x for x in vec1))
        mag2 = math.sqrt(sum(x * x for x in vec2))
        if mag1 == 0 or mag2 == 0:
            return 1.0
        dot = sum(p * q for p, q in zip(vec1, vec2))
        similarity = max(-1.0, min(1.0, dot / (mag1 * mag2)))
        return 1.0 - similarity

    def __len__(self) -> int:
        return len(self.vectors)


In [4]:
# BM25Index (from 004_bm25) — lexical search
import re
from collections import Counter


class BM25Index:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        self._index_built: bool = False
        self.k1 = k1
        self.b = b

    def _tokenize(self, text: str) -> List[str]:
        return [t for t in re.split(r"\W+", text.lower()) if t]

    def add_documents(self, documents: List[Dict[str, Any]]):
        for document in documents:
            self.add_document(document)

    def add_document(self, document: Dict[str, Any]):
        doc_tokens = self._tokenize(document["content"])
        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._doc_len.append(len(doc_tokens))
        for token in set(doc_tokens):
            self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
        self._index_built = False

    def _build_index(self):
        if not self.documents:
            return
        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        N = len(self.documents)
        self._idf = {t: math.log(((N - f + 0.5) / (f + 0.5)) + 1) for t, f in self._doc_freqs.items()}
        self._index_built = True

    def search(self, query_text: str, k: int = 1, score_normalization_factor: float = 0.1):
        if not self.documents:
            return []
        if not self._index_built:
            self._build_index()
        query_tokens = self._tokenize(query_text)

        raw_scores = []
        for i, doc in enumerate(self.documents):
            counts = Counter(self._corpus_tokens[i])
            doc_length = self._doc_len[i]
            score = 0.0
            for token in query_tokens:
                if token not in self._idf:
                    continue
                idf = self._idf[token]
                tf = counts.get(token, 0)
                num = idf * tf * (self.k1 + 1)
                denom = tf + self.k1 * (1 - self.b + self.b * (doc_length / self._avg_doc_len))
                score += num / (denom + 1e-9)
            if score > 1e-9:
                raw_scores.append((score, doc))

        raw_scores.sort(key=lambda item: item[0], reverse=True)
        results = [(doc, math.exp(-score_normalization_factor * s)) for s, doc in raw_scores[:k]]
        results.sort(key=lambda item: item[1])
        return results

    def __len__(self) -> int:
        return len(self.documents)


## `Retriever`: combining both indexes with RRF

This is the new part. `Retriever` wraps any number of indexes (each just needs `add_document(s)` and `search`) and fuses their results:

1. Query every index, asking each for more results than we actually need (`k * 5`) so there's enough overlap to fuse meaningfully.
2. For each document, record its *rank* (1st, 2nd, 3rd, ...) in each index's result list — a document an index didn't return at all counts as infinitely ranked.
3. Score each document by summing `1 / (k_rrf + rank)` across the indexes it appeared in. `k_rrf` (default 60, a standard RRF constant) softens the effect of small rank differences.
4. Sort by that combined score, descending — higher is better here, unlike the per-index distance/score conventions.


In [5]:
class Retriever:
    def __init__(self, *indexes):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)

    def add_document(self, document):
        for index in self._indexes:
            index.add_document(document)

    def add_documents(self, documents):
        for index in self._indexes:
            index.add_documents(documents)

    def search(self, query_text, k=1, k_rrf=60):
        all_results = [index.search(query_text, k=k * 5) for index in self._indexes]

        doc_ranks = {}
        for idx, results in enumerate(all_results):
            for rank, (doc, _) in enumerate(results):
                doc_id = id(doc)
                if doc_id not in doc_ranks:
                    doc_ranks[doc_id] = {"doc_obj": doc, "ranks": [float("inf")] * len(self._indexes)}
                doc_ranks[doc_id]["ranks"][idx] = rank + 1

        def rrf_score(ranks):
            return sum(1.0 / (k_rrf + r) for r in ranks if r != float("inf"))

        scored = [(v["doc_obj"], rrf_score(v["ranks"])) for v in doc_ranks.values()]
        scored = [(doc, score) for doc, score in scored if score > 0]
        scored.sort(key=lambda x: x[1], reverse=True)

        return scored[:k]


## Building and searching the hybrid index

Same `report.md`, chunked and added to both indexes through the `Retriever`. Then we search with the same ambiguous, exact-token-heavy query from `004_bm25` — but now vector search backs up BM25 for anything it might miss on wording alone.


In [6]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

vector_index = VectorIndex(embedding_fn=generate_embedding)
bm25_index = BM25Index()
retriever = Retriever(bm25_index, vector_index)

retriever.add_documents([{"content": chunk} for chunk in chunks])
print(f"Indexed {len(chunks)} chunks into both indexes")


Indexed 15 chunks into both indexes


In [7]:
query = "CTX-204b biomarker results"
results = retriever.search(query, k=2)

print(f"Query: {query}\n")
for rank, (doc, score) in enumerate(results, start=1):
    preview = doc["content"][:300].replace("\n", " ")
    print(f"--- Rank {rank} | RRF score {score:.4f} (higher is better) ---")
    print(preview + "...\n")


Query: CTX-204b biomarker results

--- Rank 1 | RRF score 0.0328 (higher is better) ---
Section 9: Pharmaceutical Development - Compound CTX-204b Phase IIa Update  Promising results emerged from the Phase IIa clinical trial (`Trial ID: CTX204b-P2A-001`) for Compound CTX-204b, our lead candidate targeting Receptor Pathway Gamma-7. Interim analysis of data from the initial patient cohort...

--- Rank 2 | RRF score 0.0318 (higher is better) ---
Section 1: Medical Research - Understanding XDR-471 Syndrome  This year saw significant strides in our understanding of XDR-471 syndrome, a rare neurodegenerative condition previously hampered by diagnostic ambiguity. The team focused on correlating clinical presentations with specific genetic marke...

